<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.5-self-hosting-with-ollama/notebooks/exercises-11_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.5: Self-Hosting with Ollama

*Module 11 · Cost Optimization & Efficiency*

> No API keys. No rate limits. No per-token billing. Ollama lets you run open-source models like Gemma, Llama, and Qwen locally — on your laptop, your server, or your GPU. This lesson installs Ollama, runs models, builds a Python client, and answers the key question: when does self-hosting actually save money?

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-11.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Install & Run — First Local LLM in 5 Minutes — `setup.sh`
2. Step 2 — Model Selection Guide — Which Model for Which Task — `model_guide.py`
3. Step 3 — Python Client — Chat, Stream, Embed, Structured Output — `ollama_client.py`
4. Step 4 — Local API Server — Same Interface as Your Cloud Gateway — `local_gateway.py`
5. Step 5 — When Self-Hosting Makes Sense — The Economics — `cost_analysis.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q fastapi uvicorn requests pydantic ollama


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Install & Run — First Local LLM in 5 Minutes

One install command. One pull command. One run command. Done.

**`setup.sh`** · _Block 1 of 5_

═══════════════════════════════════════════ — OLLAMA SETUP


In [ ]:
# ═══════════════════════════════════════════
# OLLAMA SETUP
# Install → pull model → run
# ═══════════════════════════════════════════

# ── Step 1: Install Ollama ──
# macOS / Linux
curl -fsSL https://ollama.com/install.sh | sh

# Windows: download from https://ollama.com/download

# Verify installation
ollama --version

# ── Step 2: Pull a model ──
# Small and fast (good for laptops without GPU)
ollama pull gemma3:4b              # Google Gemma 3 4B — 2.5 GB

# Medium (needs 8GB+ RAM)
ollama pull llama3.1:8b            # Meta Llama 3.1 8B — 4.7 GB

# Best quality (needs 16GB+ RAM or GPU)
ollama pull qwen3:14b              # Alibaba Qwen 3 14B — 9 GB

# Coding specialist
ollama pull qwen2.5-coder:7b       # Code generation — 4.7 GB

# Embedding model (for RAG)
ollama pull nomic-embed-text       # 274 MB — fast embeddings

# ── Step 3: Run your first query ──
ollama run gemma3:4b "What is the Model Context Protocol?"

# ── Step 4: Check what's installed ──
ollama list

# ── Step 5: Ollama runs as a server on port 11434 ──
# Test the API directly
curl http://localhost:11434/api/generate \
  -d '{"model": "gemma3:4b", "prompt": "Hello!", "stream": false}'


> **What just happened?** Ollama installed in one command. Models are pulled like Docker images — ollama pull gemma3:4b downloads the 2.5 GB model file. ollama run starts an interactive chat. Behind the scenes, Ollama runs as a local server on localhost:11434 with a REST API — the same API that Python clients and FastAPI wrappers talk to. Total time from zero to running: under 5 minutes. No API key, no signup, no billing.


### Step 2 · Model Selection Guide — Which Model for Which Task

Match the model to your hardware and use case.

**`model_guide.py`** · _Block 2 of 5_

OLLAMA MODEL SELECTION GUIDE — Match model → hardware → use case.


In [ ]:
# =============================================
# OLLAMA MODEL SELECTION GUIDE
# Match model → hardware → use case.
# =============================================

MODELS = {
    # ── Small (laptop-friendly, no GPU needed) ──
    "gemma3:1b":  {
        "size": "815 MB", "ram": "4 GB", "speed": "~50 tok/s CPU",
        "quality": 6, "best_for": "Classification, extraction, simple Q&A",
    },
    "gemma3:4b":  {
        "size": "2.5 GB", "ram": "6 GB", "speed": "~30 tok/s CPU",
        "quality": 7.5, "best_for": "General chat, summarization, translation",
    },
    "phi4-mini:3.8b": {
        "size": "2.5 GB", "ram": "6 GB", "speed": "~35 tok/s CPU",
        "quality": 7.5, "best_for": "Reasoning, math, structured output",
    },

    # ── Medium (needs 16GB RAM or GPU) ──
    "llama3.1:8b": {
        "size": "4.7 GB", "ram": "10 GB", "speed": "~15 tok/s CPU, ~80 tok/s GPU",
        "quality": 8, "best_for": "General purpose, instruction following",
    },
    "qwen2.5-coder:7b": {
        "size": "4.7 GB", "ram": "10 GB", "speed": "~15 tok/s CPU, ~80 tok/s GPU",
        "quality": 8.5, "best_for": "Code generation, debugging, code review",
    },

    # ── Large (needs GPU or 32GB+ RAM) ──
    "qwen3:14b": {
        "size": "9 GB", "ram": "18 GB", "speed": "~5 tok/s CPU, ~60 tok/s GPU",
        "quality": 8.5, "best_for": "Analysis, complex reasoning, multilingual",
    },
    "llama3.3:70b": {
        "size": "43 GB", "ram": "48 GB", "speed": "GPU only, ~30 tok/s",
        "quality": 9, "best_for": "Near-GPT-4 quality for all tasks",
    },

    # ── Embeddings ──
    "nomic-embed-text": {
        "size": "274 MB", "ram": "2 GB", "speed": "~500 embeds/s",
        "quality": 8, "best_for": "RAG embeddings, semantic search, clustering",
    },
}

print("Ollama Model Guide:\n")
print(f"  {'Model':22s} {'Size':>7s}  {'RAM':>6s}  {'Quality':>7s}  Best for")
print(f"  {'─'*85}")
for name, info in MODELS.items():
    print(f"  {name:22s} {info['size']:>7s}  {info['ram']:>6s}  {info['quality']:>5}/10  {info['best_for']}")

print("""
  Hardware recommendations:
    8 GB RAM laptop (no GPU):  gemma3:4b or phi4-mini
    16 GB RAM laptop:          llama3.1:8b or qwen2.5-coder:7b
    32 GB RAM or GPU:          qwen3:14b
    48 GB+ or multi-GPU:       llama3.3:70b
""")


> **What just happened?** A practical model guide matched to real hardware: 8 GB laptop → Gemma 3 4B or Phi-4 Mini (both ~2.5 GB, 30-35 tok/s on CPU, quality 7.5/10). 16 GB laptop → Llama 3.1 8B or Qwen 2.5 Coder (quality 8-8.5/10). GPU or 32 GB+ → Qwen 3 14B (quality 8.5, good multilingual). For RAG → nomic-embed-text (274 MB, 500 embeddings/second). Key rule: the model must fit in RAM. If it doesn't, it swaps to disk and becomes 100x slower.


### Step 3 · Python Client — Chat, Stream, Embed, Structured Output

The ollama library gives you the same API patterns you know from Gemini/OpenAI.

**`ollama_client.py`** · _Block 3 of 5_

OLLAMA PYTHON CLIENT — Chat, stream, embed, structured output.


In [ ]:
# =============================================
# OLLAMA PYTHON CLIENT
# Chat, stream, embed, structured output.
# pip install ollama
# =============================================

import ollama
import json, time

# ── 1. Basic chat ──
def basic_chat(prompt: str, model: str = "gemma3:4b") -> str:
    """Simple chat — send prompt, get response."""
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]

# ── 2. Streaming chat ──
def streaming_chat(prompt: str, model: str = "gemma3:4b"):
    """Stream tokens — same UX as ChatGPT."""
    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    full_response = []
    for chunk in stream:
        token = chunk["message"]["content"]
        print(token, end="", flush=True)
        full_response.append(token)
    print()
    return "".join(full_response)

# ── 3. System prompt + multi-turn ──
def chat_with_system(system: str, messages: list[dict],
                      model: str = "gemma3:4b") -> str:
    """Chat with a system prompt and conversation history."""
    full_messages = [{"role": "system", "content": system}] + messages
    response = ollama.chat(model=model, messages=full_messages)
    return response["message"]["content"]

# ── 4. Embeddings ──
def get_embedding(text: str, model: str = "nomic-embed-text") -> list[float]:
    """Generate embeddings locally — free, private, fast."""
    response = ollama.embed(model=model, input=text)
    return response["embeddings"][0]

# ── 5. Structured JSON output ──
def structured_output(prompt: str, model: str = "gemma3:4b") -> dict:
    """Force JSON output from the model."""
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt + "\n\nRespond ONLY in valid JSON."}],
        format="json",  # ← forces JSON output mode
    )
    return json.loads(response["message"]["content"])

# ── Demo ──
print("1. Basic chat:")
print(f"   {basic_chat('What is Python? One sentence.')}\n")

print("2. Streaming:")
print("   ", end="")
streaming_chat("Count from 1 to 5.")

print("\n3. System prompt:")
resp = chat_with_system(
    system="You are a pirate. Respond in pirate speak.",
    messages=[{"role": "user", "content": "What is machine learning?"}],
)
print(f"   {resp[:100]}...\n")

print("4. Embeddings:")
emb = get_embedding("Hello world")
print(f"   Dimension: {len(emb)}, first 5: {emb[:5]}\n")

print("5. Structured JSON:")
result = structured_output("List 3 programming languages with their main use case.")
print(f"   {json.dumps(result, indent=2)[:200]}")


> **What just happened?** Five capabilities with the ollama Python library: (1) Basic chat — same pattern as Gemini's generate_content() . (2) Streaming — stream=True yields tokens one at a time. (3) System prompt + multi-turn — system role + conversation history. (4) Embeddings — ollama.embed() with nomic-embed-text, returns 768-dim vectors locally. (5) Structured JSON — format="json" forces valid JSON output. The interface is nearly identical to cloud APIs — switching between Ollama and Gemini requires changing ~3 lines of code.


### Step 4 · Local API Server — Same Interface as Your Cloud Gateway

Wrap Ollama in FastAPI with the same /v1/chat endpoint as Module 10.3's gateway.

**`local_gateway.py`** · _Block 4 of 5_

LOCAL API SERVER — Same /v1/chat interface as the cloud gateway.


In [ ]:
# =============================================
# LOCAL API SERVER
# Same /v1/chat interface as the cloud gateway.
# Swap between cloud and local by changing URL.
# =============================================

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import ollama, json, time

app = FastAPI(title="Netsetos Local AI Gateway")

class ChatRequest(BaseModel):
    model: str = "gemma3:4b"
    messages: list[dict]
    stream: bool = False
    temperature: float = 0.7

@app.post("/v1/chat")
async def chat(req: ChatRequest):
    start = time.time()
    
    if req.stream:
        return StreamingResponse(
            _stream_response(req), media_type="text/event-stream",
            headers={"Cache-Control": "no-cache"})
    
    response = ollama.chat(
        model=req.model,
        messages=req.messages,
        options={"temperature": req.temperature},
    )
    
    latency = round((time.time() - start) * 1000)
    
    return {
        "text": response["message"]["content"],
        "model": req.model,
        "provider": "ollama-local",
        "latency_ms": latency,
        "cost_usd": 0.0,  # ← local = free!
        "usage": {
            "input_tokens": response.get("prompt_eval_count", 0),
            "output_tokens": response.get("eval_count", 0),
        },
    }

async def _stream_response(req: ChatRequest):
    stream = ollama.chat(model=req.model, messages=req.messages, stream=True,
                          options={"temperature": req.temperature})
    for chunk in stream:
        token = chunk["message"]["content"]
        yield f"data: {json.dumps({'type': 'token', 'text': token})}\n\n"
    yield f"data: {json.dumps({'type': 'done'})}\n\n"

@app.post("/v1/embed")
async def embed(request: dict):
    text = request.get("text", "")
    model = request.get("model", "nomic-embed-text")
    response = ollama.embed(model=model, input=text)
    return {"embedding": response["embeddings"][0], "model": model, "cost_usd": 0.0}

@app.get("/health")
async def health():
    models = ollama.list()
    return {"status": "healthy", "provider": "ollama",
            "models": [m["model"] for m in models.get("models", [])]}

print("""
Run locally:
  uvicorn local_gateway:app --host 0.0.0.0 --port 8080

Test:
  curl -X POST http://localhost:8080/v1/chat \\
    -H "Content-Type: application/json" \\
    -d '{"messages": [{"role": "user", "content": "Hello!"}]}'

Switch your client:
  CLOUD:  base_url = "https://ai-gateway.run.app"
  LOCAL:  base_url = "http://localhost:8080"
  # Everything else stays the same — same /v1/chat endpoint!
""")


> **What just happened?** A FastAPI server wrapping Ollama with the exact same /v1/chat interface as the cloud gateway from 10.3. Chat, streaming (SSE), and embeddings — all endpoints identical. The response includes "cost_usd": 0.0 because local = free. Your client code switches between cloud and local by changing one URL : https://ai-gateway.run.app → http://localhost:8080 . Same request body, same response format, zero code changes.


### Step 5 · When Self-Hosting Makes Sense — The Economics

The real question: is running your own model cheaper than paying per token?

**`cost_analysis.py`** · _Block 5 of 5_

SELF-HOST vs API COST COMPARISON — When does owning beat renting?


In [ ]:
# =============================================
# SELF-HOST vs API COST COMPARISON
# When does owning beat renting?
# =============================================

def cost_comparison(daily_requests: int, avg_tokens: int = 1000):
    """Compare monthly costs: API vs self-hosted."""
    
    # ── API costs (per month) ──
    api_costs = {
        "Gemini Flash": {
            "per_1m_in": 0.10, "per_1m_out": 0.40,
            "monthly": daily_requests * avg_tokens * 0.25 / 1e6 * 30,  # blended rate
        },
        "Gemini Pro": {
            "per_1m_in": 1.25, "per_1m_out": 10.0,
            "monthly": daily_requests * avg_tokens * 5.0 / 1e6 * 30,
        },
    }
    
    # ── Self-hosted costs (per month) ──
    self_hosted = {
        "Laptop (existing)": {
            "model": "gemma3:4b", "hardware": "₹0 (already own it)",
            "electricity": 500,   # ₹/month estimate
            "monthly_inr": 500,
            "quality": "7.5/10", "throughput": "~30 tok/s",
            "limit": "1 concurrent user",
        },
        "Cloud GPU (L4)": {
            "model": "llama3.1:8b", "hardware": "GCP g2-standard-8",
            "monthly_inr": 36000,  # ~$430/month for L4 GPU
            "quality": "8/10", "throughput": "~80 tok/s",
            "limit": "~50 concurrent users",
        },
        "Cloud GPU (A100)": {
            "model": "llama3.3:70b", "hardware": "GCP a2-highgpu-1g",
            "monthly_inr": 200000,  # ~$2,400/month for A100
            "quality": "9/10", "throughput": "~30 tok/s",
            "limit": "~100 concurrent users",
        },
    }
    
    print(f"  Cost comparison at {daily_requests:,} requests/day ({avg_tokens} tokens avg):\n")
    
    print("  ── API Costs ──")
    for name, info in api_costs.items():
        monthly_inr = info["monthly"] * 84
        print(f"    {name:20s}: ₹{monthly_inr:>8,.0f}/month (${info['monthly']:.2f})")
    
    print("\n  ── Self-Hosted Costs ──")
    for name, info in self_hosted.items():
        print(f"    {name:20s}: ₹{info['monthly_inr']:>8,}/month  [{info['model']}, {info['quality']}, {info['throughput']}]")
    
    # Find the crossover point
    flash_monthly = api_costs["Gemini Flash"]["monthly"] * 84
    print(f"\n  ── Verdict at {daily_requests:,} req/day ──")
    if flash_monthly < 500:
        print(f"    ✅ API wins: Flash costs ₹{flash_monthly:,.0f}/month — cheaper than electricity")
    elif flash_monthly < 36000:
        print(f"    🤔 Borderline: Flash = ₹{flash_monthly:,.0f}. Laptop self-hosting saves money but limited throughput.")
    else:
        print(f"    💰 Self-host wins: Flash = ₹{flash_monthly:,.0f}, but cloud GPU = ₹36,000 with 50 concurrent users.")

print("Cost Analysis:\n")
cost_comparison(1000)    # startup scale
print()
cost_comparison(50000)   # growth scale
print()
cost_comparison(500000)  # enterprise scale


> **What just happened?** Cost comparison at three scales: (1) 1,000 req/day: Gemini Flash ≈ ₹630/month — API wins (cheaper than electricity for self-hosting). (2) 50,000 req/day: Flash ≈ ₹31,500/month — borderline. A laptop with Gemma 4B costs ₹500/month but can't handle 50K req/day. A cloud L4 GPU costs ₹36,000 with 50 concurrent users. (3) 500,000 req/day: Flash ≈ ₹3,15,000/month — self-hosting wins massively. One L4 GPU at ₹36,000 handles most of the load.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
